# Frank-Wolfe 法デモ: Sioux Falls ネットワーク

Frank-Wolfe (Conditional Gradient) 法による静的利用者均衡配分の実行例。

Beckmann の等価最小化問題:

$$\min_{f} Z(f) = \sum_a \int_0^{f_a} t_a(x) \, dx$$

**参考文献:**
- LeBlanc, L. J. et al. (1975). An efficient approach to solving the road network equilibrium traffic assignment problem. *Transportation Research*, 9(5), 309-318.
- Sheffi, Y. (1985). *Urban Transportation Networks*. Prentice-Hall.

## 1. データの読み込み

In [ ]:
from network import Network
from cost_function import bpr_cost, bpr_cost_derivative, bpr_cost_integral
from frank_wolfe import FrankWolfe

# Sioux Falls ネットワーク (GMNS Plus 形式)
data_dir = "../itapas/data/SiouxFalls"
net = Network(f"{data_dir}/node.csv", f"{data_dir}/link.csv", f"{data_dir}/demand.csv")

print(f"ノード数: {net.num_nodes}")
print(f"リンク数: {net.num_links}")
print(f"OD対数:   {len(net.demand)}")
print(f"総需要:   {sum(net.demand.values()):.0f}")

## 2. Frank-Wolfe 法の実行

In [ ]:
solver = FrankWolfe(net, bpr_cost, bpr_cost_derivative, bpr_cost_integral)

link_flow, log = solver.solve(max_iter=200, gap_threshold=1e-3, verbose=True)

## 3. 収束過程の可視化

In [ ]:
import matplotlib.pyplot as plt

iters = [e['iter'] for e in log]
rel_gaps = [e['rel_gap'] for e in log]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(iters, rel_gaps, 'o-', markersize=2)
ax.set_xlabel("Iteration")
ax.set_ylabel("Relative Duality Gap")
ax.set_title("Frank-Wolfe Convergence (Sioux Falls Network)")
ax.grid(True, alpha=0.3)
ax.axhline(y=1e-3, color='r', linestyle='--', label='Threshold (1e-3)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. 結果の確認

In [ ]:
import pandas as pd
import numpy as np

link_df = pd.read_csv(f"{data_dir}/link.csv")
results = pd.DataFrame({
    'link_id': link_df['link_id'].values,
    'from': link_df['from_node_id'].values,
    'to': link_df['to_node_id'].values,
    'flow': link_flow,
    'capacity': net.link_capacity,
    'V/C': link_flow / net.link_capacity,
    'fftt': net.link_fftt,
    'travel_time': bpr_cost(link_flow, net.link_fftt, net.link_capacity,
                            net.link_alpha, net.link_beta)
})

print(f"総リンクフロー: {link_flow.sum():.0f}")
print(f"最大 V/C 比:    {results['V/C'].max():.3f}")
print(f"平均 V/C 比:    {results['V/C'].mean():.3f}")
print()
print("V/C比上位10リンク:")
results.sort_values('V/C', ascending=False).head(10)